# Household Segmentation on Set-Top-Box Viewing Data

**Distributed Database Management, Technion — Project 2**

## Problem

A television provider holds two large tables: household demographics, and a log
of what each set-top box watched. Neither on its own says much. Joined and
segmented, they answer the question the provider actually has — *which kinds of
household watch which kinds of content*, and can that be tracked as events
arrive rather than overnight.

## Data

Two Parquet tables from the FWM set-top-box dataset, read from the course mount:

| Table | Contents |
|---|---|
| `Project2_demographic.parquet` | One row per household: income, household size, age bands, region and other categorical attributes |
| `Project2_static_viewing_data.parquet` | One row per viewing event: household, station, programme, timing |

## Goals

1. Turn mixed categorical and numerical demographics into a feature vector that
   K-Means can use without one attribute dominating by scale alone.
2. Segment households, and check the segments are real rather than an artefact
   of the chosen `k`.
3. Characterise each segment by what it watches.
4. Repeat the analysis over a stream, so segmentation is a running view rather
   than a one-off batch job.

## Method

Preprocessing → SVD/PCA projection → K-Means (k = 8) → per-cluster viewing
profiles → the same pipeline over a Kafka stream.

> **Note.** This notebook was written against Databricks: it reads from `/dbfs`
> mounts, uses `display()` for tables, and connects to a course Kafka topic.
> It is preserved as a record of the work and will not run outside that
> environment. Full write-up in [`report.pdf`](report.pdf).

In [ ]:
from pyspark.sql.types import *
import pyspark.sql.functions as F
import os, time
from pyspark import SparkContext
from pyspark.sql import SparkSession
import re
from pyspark.sql.window import Window 
from pyspark.sql import SparkSession 
from pyspark.sql.functions import split, explode, count, avg, col, when
 
spark = SparkSession.builder.appName("my_project_2").getOrCreate()
sc = spark.sparkContext 

## 1 · Demographics

One row per household. Everything here is a candidate segmentation feature, and
most of it is categorical — which is what makes the encoding step below matter.

In [ ]:
demographic_df = spark.read.parquet("/dbfs/mnt/coursedata2024/fwm-stb-data/Project2_demographic.parquet")
demographic_df.printSchema()
display(demographic_df.limit(10))

## 2 · Viewing events

The batch half of the viewing log. Joined to the segments later to describe what
each cluster actually watches; the streaming section at the end replays the same
shape of data as events.

In [ ]:
static_viewing_df = spark.read.parquet("/dbfs/mnt/coursedata2024/fwm-stb-data/Project2_static_viewing_data.parquet")
static_viewing_df.printSchema()
display(static_viewing_df.limit(10))

# Part A · Batch analysis

## 3 · Feature extraction

Categorical columns go through `StringIndexer` → `OneHotEncoder`; numerical
columns are normalised. Both steps are necessary for a distance-based method:
K-Means minimises Euclidean distance, so an un-normalised attribute with a wide
range would dominate the objective purely because of its units, and an integer
index over categories would invent an ordering ("region 3 is between region 2
and region 4") that does not exist. `VectorAssembler` combines the result into
the single `features` column Spark ML expects.

In [ ]:

from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml import Pipeline
from pyspark.sql.functions import max, lit, min
from pyspark.ml.feature import VectorAssembler



column_order = [
    'household_id', 'household_size', 'num_adults', 'num_generations',
    'marital_status', 'race_code', 'dwelling_type', 'home_owner_status',
    'length_residence', 'home_market_value', 'net_worth', 'education_highest',
    'gender_individual'
]

categorical_columns_list=['marital_status','race_code','dwelling_type','home_owner_status','gender_individual' ,'education_highest']

indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_index") for c in categorical_columns_list]
encoders = [OneHotEncoder(inputCol=f"{c}_index", outputCol=f"{c}_ohe") for c in categorical_columns_list]
pipeline = Pipeline(stages=indexers + encoders)
model = pipeline.fit(demographic_df)
demographic_df_encoded = model.transform(demographic_df) 

# Drop index + original categorical_columns_list columns
columns_to_drop = categorical_columns_list + [f"{c}_index" for c in categorical_columns_list]
demographic_df_encoded = demographic_df_encoded.drop(*columns_to_drop)

for categorical_column in categorical_columns_list:
    demographic_df_encoded = demographic_df_encoded.withColumnRenamed(f"{categorical_column}_ohe", categorical_column)


demographic_df_encoded = demographic_df_encoded.select(column_order)

columns_to_normalize = ['num_generations', 'household_size', 'num_adults' ,'num_generations', 'length_residence', 'home_market_value', 'net_worth']

# Normalize each column
for column in columns_to_normalize:
    col_min = demographic_df_encoded.select(min(col(column))).collect()[0][0]
    col_max = demographic_df_encoded.select(max(col(column))).collect()[0][0]
    
    demographic_df_encoded = demographic_df_encoded.withColumn(
        column,
        (col(column) - lit(col_min)) / (lit(col_max) - lit(col_min))
    )

all_feature_columns = [
    'household_size', 'num_adults', 'num_generations',
    'marital_status', 'race_code', 'dwelling_type', 'home_owner_status',
    'length_residence', 'home_market_value', 'net_worth', 'education_highest',
    'gender_individual'
]

# Create the VectorAssembler
assembler = VectorAssembler(inputCols=all_feature_columns, outputCol='features')
# Apply the VectorAssembler to the DataFrame
demographic_df_features = assembler.transform(demographic_df_encoded)
# Show the result

demographic_df_features=demographic_df_features.select('household_id', 'features')

display(demographic_df_features.limit(10)) 


## 4 · Projection

Singular value decomposition onto two dimensions, to see whether the households
separate at all before committing to a number of clusters. The projection keeps
the directions of largest variance; three groupings are visible, which is the
first evidence that segmentation is worth doing rather than an assumption.

In [ ]:
from pyspark.ml.linalg import Vectors as ML_Vectors
from pyspark.ml.recommendation import ALS
from pyspark.ml import Estimator
import matplotlib.pyplot as plt 
from pyspark.mllib.linalg import Vectors as MLLib_Vectors
from pyspark.mllib.linalg.distributed import RowMatrix

# Convert the 'features' column to an RDD of MLLib Dense Vectors
features_rdd = demographic_df_features.select("features").rdd.map(lambda row: MLLib_Vectors.dense(row['features'].toArray()))

# Create a RowMatrix from the RDD
row_matrix = RowMatrix(features_rdd)

# Apply SVD with k=2
svd = row_matrix.computeSVD(2, computeU=True) 

# Extract U (the reduced data matrix)
U = svd.U

U_df = U.rows.map(lambda row: row.toArray().tolist()).toDF(['U1', 'U2']) 

display(U_df.limit(10))

U_pandas_df = U_df.toPandas()
plt.figure(figsize=(10, 6))
plt.scatter(U_pandas_df['U1'], U_pandas_df['U2'])
plt.xlabel('U1')
plt.ylabel('U2')
plt.title('SVD Result: U1 and U2')
plt.show()


## 5 · K-Means

`k = 8` with a fixed seed, so the run is reproducible. Each household is then
given its Euclidean distance from its own centroid — a per-row measure of how
typical it is of its segment, which the next two sections both depend on.

In [ ]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.linalg import Vectors 
from pyspark.sql import SparkSession 
from pyspark.sql.types import FloatType 
import numpy as np
import pandas as pd

kmeans = KMeans(k=8, seed=7)  # k=8 for 8 clusters
model = kmeans.fit(demographic_df_features.select('features'))
demographic_df_with_clusters = model.transform(demographic_df_features)


# Define a UDF to calculate Euclidean distance between two vectors
def euclidean_distance(v1, v2):
    return float(sum((a - b) ** 2 for a, b in zip(v1, v2)) ** 0.5)

# Register the UDF
distance_udf = F.udf(euclidean_distance, FloatType())

# Get cluster centers as a DataFrame
centroids = model.clusterCenters()
centroids_df = spark.createDataFrame([(i, Vectors.dense(center)) for i, center in enumerate(centroids)], ["cluster", "centroid"])

# Join the original DataFrame with the centroids DataFrame on the cluster ID
demographic_df_with_centroids = demographic_df_with_clusters.join(
    centroids_df,
    demographic_df_with_clusters["prediction"] == centroids_df["cluster"],
    how="left"
).drop("prediction")

# Calculate the distance of each household from its centroid
demographic_df_with_distances = demographic_df_with_centroids.withColumn(
    "distance_from_centroid",
    distance_udf(demographic_df_with_centroids["features"], demographic_df_with_centroids["centroid"])
)

demographic_df_with_cluster_distances = demographic_df_with_distances.drop('centroid')

# Show the first few rows of the updated DataFrame
display(demographic_df_with_cluster_distances.limit(10))


## 6 · Cluster visualisation

The same 2-D projection as before, now coloured by assigned cluster. This is the
check on step 4: if the eight clusters were an artefact of the chosen `k` rather
than structure in the data, they would show as arbitrary slices of one cloud
instead of coherent regions.

In [ ]:
from pyspark.ml.feature import PCA
import seaborn as sns


# Apply PCA to reduce dimensions to 2
pca = PCA(k=2, inputCol="features", outputCol="pca_features")
pca_model = pca.fit(demographic_df_with_cluster_distances)
pca_result = pca_model.transform(demographic_df_with_cluster_distances)

display(pca_result.limit(10))

# Convert the Spark DataFrame to a Pandas DataFrame
pca_pdf = pca_result.select("household_id", "cluster", "distance_from_centroid", "pca_features").toPandas()

# Extract PCA features into separate columns
pca_features = pd.DataFrame(pca_pdf['pca_features'].tolist(), columns=['pca1', 'pca2'])
pca_pdf = pca_pdf.drop(columns=['pca_features'])
pca_pdf = pd.concat([pca_pdf, pca_features], axis=1)

# Plot using Seaborn
plt.figure(figsize=(10, 6))
sns.scatterplot(data=pca_pdf, x="pca1", y="pca2", hue="cluster", palette="tab10", marker='o', s=20)
plt.title('PCA of Clusters')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.legend(title='Cluster')
plt.show()


## 7 · Stratified subsets

Households are ranked *within* each cluster by distance from the centroid, then
sampled every 7th and every 11th row.

Ranking within cluster before sampling is the point: it keeps each subset
proportional to its cluster and spans the full range from typical to atypical
household, which uniform random sampling over the whole table would not
guarantee. Two coprime strides give two subsets that overlap only every 77th
row.

In [ ]:
from pyspark.sql import Window

window_spec = Window.partitionBy("cluster").orderBy("distance_from_centroid")

demographic_df_with_row_number = demographic_df_with_cluster_distances.withColumn(
    "row_number", F.row_number().over(window_spec) 
)

sevenths_subset = demographic_df_with_row_number.filter(
    (col("row_number") % 7) == 0
).drop("row_number")

eleven_subset = demographic_df_with_row_number.filter(
    (col("row_number") % 11) == 0 
).drop("row_number")



## 8 · What each segment watches

Join the segments back to the viewing log and aggregate per station, as a
percentage of the cluster's own viewing rather than a raw count — otherwise a
large cluster looks like a heavy viewer of everything. This is where the
demographic segmentation turns into a statement about content.

In [ ]:

def add_viewing_counts_percent(input_df): 
    joined_df = input_df.join(static_viewing_df, on = 'household_id')
    viewing_counts_per_station = joined_df.groupBy('cluster', 'station_num').agg(F.count("*").alias("view_count"))

    viewing_counts_per_cluster = viewing_counts_per_station.groupBy('cluster').agg(F.sum('view_count').alias('view_count_cluster')) 

    viewing_counts_with_totals = viewing_counts_per_station.join( 
        viewing_counts_per_cluster,
        on='cluster', 
        how='inner'
    )
    viewing_counts_percent = viewing_counts_with_totals.withColumn( 
        'percent_view_count',
        (F.col('view_count') / F.col('view_count_cluster')) * 100
    )

    viewing_counts_percent = viewing_counts_percent.drop('view_count_cluster','view_count')
    return viewing_counts_percent 

viewing_counts_per_station = static_viewing_df.groupBy('station_num').agg(F.count("*").alias("view_count"))

total_viewing_counts = viewing_counts_per_station.agg(F.sum('view_count').alias('total_view_count')).collect()[0]['total_view_count'] 

viewing_counts_percent_total = viewing_counts_per_station.withColumn(  
    'percent_view_count_total',
    (F.col('view_count') / total_viewing_counts) * 100
) 

demographic_df_percent = add_viewing_counts_percent(demographic_df_with_row_number) 
sevenths_subset_percent = add_viewing_counts_percent(sevenths_subset) 
eleven_subset_percent = add_viewing_counts_percent(eleven_subset)

def add_diff_rank(input_df):
    joined_df = input_df.join(
        viewing_counts_percent_total,
        on='station_num',
        how='inner'
    )
    df_diff_rank = joined_df.withColumn(
        'diff_rank',
        F.col('percent_view_count') - F.col('percent_view_count_total')
    )
    df_diff_rank = df_diff_rank.drop('percent_view_count' ,'percent_view_count_total')
    return df_diff_rank

demographic_df_diff_rank = add_diff_rank(demographic_df_percent) 
sevenths_subset_diff_rank = add_diff_rank(sevenths_subset_percent) 
eleven_subset_diff_rank = add_diff_rank(eleven_subset_percent) 

def get_top_10_stations_per_subset(subset_df):
    window_spec = Window.partitionBy("cluster").orderBy(F.desc("diff_rank"))

    subset_df = subset_df.withColumn(
        "row_number", F.row_number().over(window_spec)
    )
    top_10_stations = subset_df.filter(F.col('row_number') <= 10).drop('row_number')

    return top_10_stations 

sevenths_subset_diff_rank_top10 = get_top_10_stations_per_subset(sevenths_subset_diff_rank)
eleven_subset_diff_rank_top10 = get_top_10_stations_per_subset(eleven_subset_diff_rank)
demographic_df_diff_rank_top10 = get_top_10_stations_per_subset(demographic_df_diff_rank) 

for c in range(8):
    displayHTML(f"<h2>top-10 highest ’diff rank’ stations for sevenths_subset of cluster {c}</h2>")
    display(sevenths_subset_diff_rank_top10.filter(col('cluster') == c)) 
    displayHTML(f"<h2>top-10 highest ’diff rank’ stations for eleven_subset of cluster {c}</h2>")
    display(eleven_subset_diff_rank_top10.filter(col('cluster') == c))
    displayHTML(f"<h2>top-10 highest ’diff rank’ stations for full data of cluster {c}</h2>")
    display(demographic_df_diff_rank_top10.filter(col('cluster') == c))

# Convert PySpark DataFrames to Pandas DataFrames 
sevenths_df = sevenths_subset_diff_rank_top10.toPandas()
eleven_df = eleven_subset_diff_rank_top10.toPandas()
full_df = demographic_df_diff_rank_top10.toPandas()

# Set up the plot
fig, axes = plt.subplots(8, 3, figsize=(24, 40))
fig.suptitle("Top 10 Stations by 'Diff Rank' for Each Cluster and Subset", fontsize=20)

# Define colors for each subset 
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']  # Blue, Orange, Green 

# Iterate through clusters 
for cluster in range(0, 8):
    # Get data for each subset 
    sevenths_data = sevenths_df[sevenths_df['cluster'] == cluster].sort_values('diff_rank', ascending=True)
    eleven_data = eleven_df[eleven_df['cluster'] == cluster].sort_values('diff_rank', ascending=True)
    full_data = full_df[full_df['cluster'] == cluster].sort_values('diff_rank', ascending=True)
    
    # Plot data for each subset
    ax = axes[cluster-1, 0]
    ax.barh(sevenths_data['station_num'], sevenths_data['diff_rank'], color=colors[0], alpha=0.7)
    ax.set_title(f"Cluster {cluster} - Sevenths Subset")
    ax.set_xlabel('Diff Rank')
    ax.set_ylabel('Station Number')
    
    ax = axes[cluster-1, 1]
    ax.barh(eleven_data['station_num'], eleven_data['diff_rank'], color=colors[1], alpha=0.7)
    ax.set_title(f"Cluster {cluster} - Eleven Subset")
    ax.set_xlabel('Diff Rank')
    ax.set_ylabel('Station Number')
    
    ax = axes[cluster-1, 2]
    ax.barh(full_data['station_num'], full_data['diff_rank'], color=colors[2], alpha=0.7)
    ax.set_title(f"Cluster {cluster} - Full Data")
    ax.set_xlabel('Diff Rank')
    ax.set_ylabel('Station Number')

# Adjust layout and display
plt.tight_layout()
plt.subplots_adjust(top=0.95)
plt.show()

# Part B · Streaming

The batch pipeline answers the question once, over yesterday's data. This part
runs the equivalent over a Kafka topic with Spark Structured Streaming: events
arrive with an explicit schema, are aggregated in windows, and the viewing
profile becomes a continuously updated view instead of a report.

The value is not that streaming is faster — it is that segment membership and
viewing behaviour both drift, and a nightly batch cannot show the drift while it
is happening.

In [ ]:
from pyspark.sql import SparkSession, functions as F, Window
import time

# Kafka and Spark Configuration 
SCHEMA = "device_id STRING, event_date INT, event_time INT, station_num STRING, prog_code STRING, household_id STRING"
kafka_server = "kafka.eastus.cloudapp.azure.com:29092" 
topic = "view_data" 
OFFSETS_PER_TRIGGER = 50000

# Create Spark session
spark = SparkSession.builder.appName("StreamingViewingData").getOrCreate() 

# Read Streaming Data from Kafka
streaming_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", kafka_server) \
    .option("subscribe", topic) \
    .option("startingOffsets", "earliest") \
    .option("failOnDataLoss", False) \
    .option("maxOffsetsPerTrigger", OFFSETS_PER_TRIGGER) \
    .load() \
    .select(F.from_csv(F.decode("value", "US-ASCII"), schema=SCHEMA).alias("value")) \
    .select("value.*") 

global_df = None    
results = {}                                                      


# Function to process each batch
def process_batch(batch_df, epoch_id): 
    if (epoch_id >= 3): 
        streaming_query.stop() 
        return 
    # Convert household_id to string if necessary
    batch_df = batch_df.withColumn("household_id", F.col("household_id").cast("long"))
    global global_df
    if global_df is None: 
        global_df = batch_df
    else:
        global_df = global_df.union(batch_df) 
    
    
    # Function to calculate viewing percentages per cluster 
    def add_viewing_counts_percent(input_df):
        joined_df = input_df.join(global_df, on='household_id')
        viewing_counts_per_station = joined_df.groupBy('cluster', 'station_num').agg(F.count("*").alias("view_count"))

        viewing_counts_per_cluster = viewing_counts_per_station.groupBy('cluster').agg(F.sum('view_count').alias('view_count_cluster')) 

        viewing_counts_with_totals = viewing_counts_per_station.join( 
            viewing_counts_per_cluster,
            on='cluster', 
            how='inner' 
        ) 
        viewing_counts_percent = viewing_counts_with_totals.withColumn( 
            'percent_view_count', 
            (F.col('view_count') / F.col('view_count_cluster')) * 100  
        ) 

        return viewing_counts_percent.drop('view_count_cluster', 'view_count') 

    # Calculate percentage viewing for the current batch
    viewing_counts_percent_total = global_df.groupBy('station_num').agg(
        F.count("*").alias("view_count") 
    ).withColumn(
        'percent_view_count_total',
        (F.col('view_count') / F.sum('view_count').over(Window.partitionBy())) * 100
    ).drop('view_count')

    # Add diff_rank calculation
    def add_diff_rank(input_df):
        joined_df = input_df.join(
            viewing_counts_percent_total, 
            on='station_num',
            how='inner'
        )
        df_diff_rank = joined_df.withColumn(
            'diff_rank',
            F.col('percent_view_count') - F.col('percent_view_count_total') 
        )
        return df_diff_rank.drop('percent_view_count', 'percent_view_count_total')

    # Get top 10 stations per cluster based on diff_rank 
    def get_top_10_stations_per_subset(subset_df):
        window_spec = Window.partitionBy("cluster").orderBy(F.desc("diff_rank")) 
        return subset_df.withColumn(
            "row_number", F.row_number().over(window_spec)
        ).filter(F.col('row_number') <= 10).drop('row_number') 

    # Analyze the 7ths subset 
    print(f"trigger {epoch_id}, top-10 highest ’diff rank’ stations per cluster for sevenths_subset") 
    for c in range(8): 
        filtered_sevenths_subset = sevenths_subset.filter(F.col("cluster") == c) 
        sevenths_subset_percent = add_viewing_counts_percent(filtered_sevenths_subset) 
        sevenths_subset_diff_rank = add_diff_rank(sevenths_subset_percent) 
        sevenths_subset_diff_rank_top10 = get_top_10_stations_per_subset(sevenths_subset_diff_rank)
        results[(epoch_id, c)] = sevenths_subset_diff_rank_top10
        sevenths_subset_diff_rank_top10.show(truncate=False) 

# Set up the streaming query to process each batch 
update_streamig_df = streaming_df.withColumn("household_id", F.concat(F.lit("0"), streaming_df["household_id"]))
streaming_query = streaming_df.writeStream \
    .foreachBatch(process_batch) \
    .outputMode("append").start() 

# Run the stream for at least 3 batches/triggers
streaming_query.awaitTermination()  # Adjust the time to ensure at least 3 triggers 



        # ---------------Set up the plot--------------
# fig, axes = plt.subplots(8, 3, figsize=(24, 40))
# fig.suptitle("Top 10 Stations by 'Diff Rank' for Each Batch", fontsize=20)

# colors = ['#1f77b4', '#ff7f0e', '#2ca02c']  # Blue, Orange, Green 

# for cluster in range(0, 8):
#         first_batch = results[(0, cluster)].toPandas()
#         second_batch = results[(1, cluster)].toPandas()
#         third_batch = results[(2, cluster)].toPandas()


#         first_batch_sorted = first_batch[first_batch['cluster'] == cluster].sort_values('diff_rank', ascending=True)
#         second_batch_sorted = second_batch[second_batch['cluster'] == cluster].sort_values('diff_rank', ascending=True)
#         third_batch_sorted = third_batch[third_batch['cluster'] == cluster].sort_values('diff_rank', ascending=True)
                
#         ax = axes[cluster-1, 0]
#         ax.barh(first_batch_sorted['station_num'], first_batch_sorted['diff_rank'], color=colors[0], alpha=0.7)
#         ax.set_title(f"Cluster {cluster} - First Batch")
#         ax.set_xlabel('Diff Rank')
#         ax.set_ylabel('Station Number')
                
#         ax = axes[cluster-1, 1]
#         ax.barh(second_batch_sorted['station_num'], second_batch_sorted['diff_rank'], color=colors[1], alpha=0.7)
#         ax.set_title(f"Cluster {cluster} - Second Batch")
#         ax.set_xlabel('Diff Rank')
#         ax.set_ylabel('Station Number')  

#         ax = axes[cluster-1, 2]
#         ax.barh(third_batch_sorted['station_num'], third_batch_sorted['diff_rank'], color=colors[2], alpha=0.7)
#         ax.set_title(f"Cluster {cluster} - Third Batch")
#         ax.set_xlabel('Diff Rank')
#         ax.set_ylabel('Station Number')

# plt.tight_layout()
# plt.subplots_adjust(top=0.95)
# plt.show()  

